# Phase 2: Deep Learning Pipeline for Infant Cry Classification

**Architecture:** CNN + BiLSTM + Temporal Attention with Domain Feature Fusion

**Key improvements over Phase 1:**
- GroupShuffleSplit by infant ID (no infant in both train and test)
- LDAM + Deferred Re-Weighting loss for extreme class imbalance
- Mel-spectrogram input with learned temporal attention
- 32-dim domain features (F0, HNR, jitter, shimmer, MFCC) fused with spectrogram branch
- Decoupled training (cRT): freeze backbone, retrain classifier with balanced sampling
- Two-stage hierarchy: hunger vs non-hunger → 4-class fine-grained
- Full ablation study (10 variants)
- Hybrid ML+DL ensemble (stacking)
- Knowledge distillation + quantization for edge deployment

**Phase 1 baseline:** SVM with MacF1 = 0.270 (all models collapsed to predicting 'hunger')

**Target:** MacF1 0.50–0.65, all classes with non-zero F1

---
## Step 0: Setup — Mount Drive & Install Dependencies

In [ ]:
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/Infant-State-Recognition-System'

if os.path.exists(os.path.join(DRIVE_PROJECT, 'src', 'phase2', 'config.py')):
    print(f'Project found on Drive: {DRIVE_PROJECT}')
else:
    from google.colab import files
    print('Upload the project zip file:')
    uploaded = files.upload()
    import zipfile
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content/drive/MyDrive/')
    print(f'Extracted to {DRIVE_PROJECT}')

PROJECT_DIR = DRIVE_PROJECT
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q librosa==0.10.1 soundfile==0.12.1 hmmlearn==0.3.0 xgboost>=2.0.0 imbalanced-learn>=0.11.0
print('Dependencies installed.')

---
## Step 1: Imports & Configuration

In [ ]:
import sys
import gc
import json
import warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from src.phase2.config import (
    CLASSES, CLASS_TO_IDX, IDX_TO_CLASS, NUM_CLASSES, CLASS_COUNTS,
    SAMPLE_RATE, DURATION, RANDOM_STATE,
    BATCH_SIZE, MAX_EPOCHS_STUDENT, MAX_EPOCHS_TEACHER,
    MODELS_DIR, RESULTS_DIR, PLOTS_DIR, METRICS_DIR,
    AUG_TARGET_PER_CLASS, CRT_EPOCHS,
)
from src.phase2.data_pipeline import (
    load_all_originals, group_split, augment_training_set,
    precompute_features, create_dataloaders, make_class_balanced_sampler,
    PrecomputedDataset, normalize_features,
)
from src.phase2.models import (
    build_student, build_teacher, build_model,
    CNNBaseline, CNNBiLSTMAttention, MultiInputCryModel,
    HierarchicalModel, count_parameters,
)
from src.phase2.losses import LDAMLoss, DRWScheduler
from src.phase2.trainer import Trainer, HierarchicalTrainer
from src.phase2.evaluation import (
    evaluate_model, plot_confusion_matrix, plot_training_history,
    save_metrics, plot_ablation_comparison, make_ablation_table,
    three_model_comparison,
)
from src.phase2.interpretability import (
    generate_gradcam_gallery, plot_attention_weights,
    domain_feature_importance,
)
from src.phase2.hybrid import HybridEnsemble, WeightedEnsemble
from src.phase2.distillation import (
    train_with_distillation, quantize_model, benchmark_inference,
)

for d in [MODELS_DIR, RESULTS_DIR, PLOTS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Classes: {CLASSES}')
print(f'Class counts (originals): {dict(zip(CLASSES, CLASS_COUNTS))}')
print(f'Max imbalance ratio: {max(CLASS_COUNTS)/min(CLASS_COUNTS):.1f}:1')

---
## Step 2: Load Original Audio & Parse Infant IDs
Only original recordings — no augmented files. Infant IDs are parsed from filenames for group splitting.

In [ ]:
print('Loading original audio files...')
print('=' * 60)
dataset = load_all_originals()

# Verify no augmented files leaked
aug_count = sum(1 for d in dataset if '_aug_' in d['filepath'])
assert aug_count == 0, f'DATA LEAK: {aug_count} augmented files found!'
print(f'Augmented files in dataset: {aug_count} (clean)')

# Show infant ID statistics
infant_ids = set(d['infant_id'] for d in dataset)
print(f'\nUnique infants: {len(infant_ids)}')
infant_class_dist = Counter(d['infant_id'] for d in dataset)
print(f'Recordings per infant: min={min(infant_class_dist.values())}, '
      f'max={max(infant_class_dist.values())}, '
      f'median={np.median(list(infant_class_dist.values())):.0f}')

---
## Step 3: GroupShuffleSplit by Infant ID (70/15/15)
No infant appears in more than one split — prevents identity-based leakage.

In [ ]:
print('Splitting by infant ID (70/15/15)...')
print('=' * 60)
train_data, val_data, test_data = group_split(dataset)

# Verify no infant leakage
train_ids = set(d['infant_id'] for d in train_data)
val_ids = set(d['infant_id'] for d in val_data)
test_ids = set(d['infant_id'] for d in test_data)
print(f'\nInfant IDs: train={len(train_ids)}, val={len(val_ids)}, test={len(test_ids)}')
print(f'Overlap train/val: {len(train_ids & val_ids)} (must be 0)')
print(f'Overlap train/test: {len(train_ids & test_ids)} (must be 0)')
print(f'Overlap val/test: {len(val_ids & test_ids)} (must be 0)')

---
## Step 4: Augment ONLY Training Set
Target: ~384 samples per class. Augmentation uses random compositions of 2-3 transforms.

In [ ]:
print(f'Augmenting training set (target: {AUG_TARGET_PER_CLASS}/class)...')
print('=' * 60)

# Save original count for no-augmentation ablation (Fix 6)
n_train_originals = len(train_data)

train_data_aug = augment_training_set(train_data, target=AUG_TARGET_PER_CLASS)

print(f'\nTraining: {len(train_data)} originals -> {len(train_data_aug)} after augmentation')
train_dist = Counter(d['label'] for d in train_data_aug)
print(f'Distribution: {dict(sorted(train_dist.items()))}')
print(f'\nVal: {len(val_data)} (untouched)')
print(f'Test: {len(test_data)} (untouched)')

---
## Step 5: Pre-extract Features
Mel-spectrograms (64 x ~438) + 32-dim domain features (F0, HNR, jitter, shimmer, MFCC, delta-MFCC).

Pre-computing saves time during training — features are cached on Drive.

In [ ]:
from src.phase2.config import FEATURES_DIR

cache_path = FEATURES_DIR / 'phase2_features.npz'

if cache_path.exists():
    print('Loading cached features from Drive...')
    cached = np.load(cache_path, allow_pickle=True)
    train_mel = cached['train_mel']
    train_dom = cached['train_dom']
    train_labels = cached['train_labels']
    val_mel = cached['val_mel']
    val_dom = cached['val_dom']
    val_labels = cached['val_labels']
    test_mel = cached['test_mel']
    test_dom = cached['test_dom']
    test_labels = cached['test_labels']
    n_train_originals = int(cached.get('n_train_originals', n_train_originals))
    print(f'Loaded: train={len(train_labels)}, val={len(val_labels)}, test={len(test_labels)}')
else:
    print('Extracting features (this takes ~20 min)...')
    print('=' * 60)
    
    print('\nTraining set:')
    train_mel, train_dom, train_labels = precompute_features(train_data_aug, desc='Train')
    
    print('\nValidation set:')
    val_mel, val_dom, val_labels = precompute_features(val_data, desc='Val')
    
    print('\nTest set:')
    test_mel, test_dom, test_labels = precompute_features(test_data, desc='Test')
    
    # Cache to Drive (include n_train_originals for ablation)
    FEATURES_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache_path,
        train_mel=train_mel, train_dom=train_dom, train_labels=train_labels,
        val_mel=val_mel, val_dom=val_dom, val_labels=val_labels,
        test_mel=test_mel, test_dom=test_dom, test_labels=test_labels,
        n_train_originals=n_train_originals,
    )
    print(f'\nFeatures cached to {cache_path}')

print(f'\nShapes:')
print(f'  Train mel: {train_mel.shape}, domain: {train_dom.shape}')
print(f'  Val mel:   {val_mel.shape},   domain: {val_dom.shape}')
print(f'  Test mel:  {test_mel.shape},  domain: {test_dom.shape}')

# Verify NaN/Inf
for name, arr in [('train_mel', train_mel), ('train_dom', train_dom),
                   ('val_mel', val_mel), ('val_dom', val_dom),
                   ('test_mel', test_mel), ('test_dom', test_dom)]:
    n_nan = np.isnan(arr).sum()
    n_inf = np.isinf(arr).sum()
    if n_nan > 0 or n_inf > 0:
        print(f'  [WARN] {name}: {n_nan} NaN, {n_inf} Inf')

# Replace any NaN/Inf with 0
for arr in [train_mel, train_dom, val_mel, val_dom, test_mel, test_dom]:
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

# Save original-only features for no-augmentation ablation
orig_train_mel = train_mel[:n_train_originals].copy()
orig_train_dom = train_dom[:n_train_originals].copy()
orig_train_labels = train_labels[:n_train_originals].copy()
print(f'\nOriginal-only train features saved: {n_train_originals} samples')

# Normalize features: fit on train, apply to all (Fix 4)
print('Normalizing features (fit on train)...')
train_mel, train_dom, val_mel, val_dom, test_mel, test_dom = normalize_features(
    train_mel, train_dom, val_mel, val_dom, test_mel, test_dom)

# Also normalize the original-only features using same stats
# (re-derive from train stats since normalize_features was already applied)
# For simplicity, normalize orig separately using its own train stats
orig_mel_mean = orig_train_mel.mean(axis=(0, 2), keepdims=True)
orig_mel_std = orig_train_mel.std(axis=(0, 2), keepdims=True) + 1e-8
orig_train_mel = (orig_train_mel - orig_mel_mean) / orig_mel_std
orig_dom_mean = orig_train_dom.mean(axis=0, keepdims=True)
orig_dom_std = orig_train_dom.std(axis=0, keepdims=True) + 1e-8
orig_train_dom = (orig_train_dom - orig_dom_mean) / orig_dom_std

del dataset, train_data, train_data_aug
gc.collect()

---
## Step 6: DL Baseline — CNN Only
Establishes the minimal deep learning baseline (no LSTM, no attention, no fusion).

In [ ]:
# Create DataLoaders for training
# Get training class counts AFTER augmentation for LDAM margins
train_class_counts = [int((train_labels == i).sum()) for i in range(NUM_CLASSES)]
print(f'Training class counts: {dict(zip(CLASSES, train_class_counts))}')

train_loader, val_loader = create_dataloaders(
    train_mel, train_dom, train_labels,
    val_mel, val_dom, val_labels,
    batch_size=BATCH_SIZE, balanced=False,
)

# Test loader (no augmentation)
from torch.utils.data import DataLoader
test_ds = PrecomputedDataset(test_mel, test_dom, test_labels, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

In [ ]:
print('=== Step 6: CNN-Only Baseline ===')
cnn_model = CNNBaseline().to(device)
print(f'Parameters: {count_parameters(cnn_model):,}')

cnn_trainer = Trainer(cnn_model, device, cls_num_list=train_class_counts,
                      use_ldam=False, max_epochs=MAX_EPOCHS_STUDENT)
cnn_history = cnn_trainer.fit(train_loader, val_loader)

cnn_metrics, cnn_preds, cnn_labels, cnn_probs = evaluate_model(
    cnn_model, test_loader, device, 'CNN_Baseline')
plot_confusion_matrix(cnn_labels, cnn_preds, 'CNN_Baseline')
plot_training_history(cnn_history, 'CNN_Baseline')
save_metrics(cnn_metrics, 'CNN_Baseline')

torch.save(cnn_model.state_dict(), MODELS_DIR / 'cnn_baseline.pt')
gc.collect()

---
## Step 7: CNN + BiLSTM + Attention (Spectrogram Only)
Adds temporal modeling via BiLSTM and learned temporal attention.

In [ ]:
print('=== Step 7: CNN + BiLSTM + Attention ===')
cba_model = CNNBiLSTMAttention().to(device)
print(f'Parameters: {count_parameters(cba_model):,}')

cba_trainer = Trainer(cba_model, device, cls_num_list=train_class_counts,
                      use_ldam=False, max_epochs=MAX_EPOCHS_STUDENT)
cba_history = cba_trainer.fit(train_loader, val_loader)

cba_metrics, cba_preds, cba_labels, cba_probs = evaluate_model(
    cba_model, test_loader, device, 'CNN_BiLSTM_Attn')
plot_confusion_matrix(cba_labels, cba_preds, 'CNN_BiLSTM_Attn')
plot_training_history(cba_history, 'CNN_BiLSTM_Attn')
save_metrics(cba_metrics, 'CNN_BiLSTM_Attn')

torch.save(cba_model.state_dict(), MODELS_DIR / 'cnn_bilstm_attn.pt')
gc.collect()

---
## Step 8: Full Model with LDAM + DRW
Same architecture but with Label-Distribution-Aware Margin loss and Deferred Re-Weighting.

In [ ]:
print('=== Step 8: CNN+BiLSTM+Attn with LDAM+DRW ===')
ldam_model = CNNBiLSTMAttention().to(device)
print(f'Parameters: {count_parameters(ldam_model):,}')

ldam_trainer = Trainer(ldam_model, device, cls_num_list=train_class_counts,
                       use_ldam=True, max_epochs=MAX_EPOCHS_STUDENT)
ldam_history = ldam_trainer.fit(train_loader, val_loader)

ldam_metrics, ldam_preds, ldam_labels, ldam_probs = evaluate_model(
    ldam_model, test_loader, device, 'LDAM_DRW')
plot_confusion_matrix(ldam_labels, ldam_preds, 'LDAM_DRW')
plot_training_history(ldam_history, 'LDAM_DRW')
save_metrics(ldam_metrics, 'LDAM_DRW')

torch.save(ldam_model.state_dict(), MODELS_DIR / 'ldam_drw.pt')
gc.collect()

---
## Step 9: Multi-Input Fusion (Full Architecture)
Fuses spectrogram branch (CNN+BiLSTM+Attention) with 32-dim domain feature branch.

In [ ]:
print('=== Step 9: Multi-Input Fusion with LDAM+DRW ===')
fusion_model = build_student().to(device)
print(f'Parameters: {count_parameters(fusion_model):,}')

fusion_trainer = Trainer(fusion_model, device, cls_num_list=train_class_counts,
                         use_ldam=True, max_epochs=MAX_EPOCHS_STUDENT)
fusion_history = fusion_trainer.fit(train_loader, val_loader)

fusion_metrics, fusion_preds, fusion_labels, fusion_probs = evaluate_model(
    fusion_model, test_loader, device, 'Fusion_LDAM')
plot_confusion_matrix(fusion_labels, fusion_preds, 'Fusion_LDAM')
plot_training_history(fusion_history, 'Fusion_LDAM')
save_metrics(fusion_metrics, 'Fusion_LDAM')

torch.save(fusion_model.state_dict(), MODELS_DIR / 'fusion_ldam.pt')
gc.collect()

---
## Step 10: Decoupled Training (cRT)
Freeze the backbone (CNN+BiLSTM+Attention), retrain only the fusion + classifier layers with class-balanced sampling.

In [ ]:
print('=== Step 10: Decoupled Training (cRT) ===')

# Create class-balanced DataLoader for cRT
crt_train_loader, _ = create_dataloaders(
    train_mel, train_dom, train_labels,
    val_mel, val_dom, val_labels,
    batch_size=BATCH_SIZE, balanced=True,
)

fusion_trainer.decoupled_retrain(crt_train_loader, val_loader, crt_epochs=CRT_EPOCHS)

crt_metrics, crt_preds, crt_labels, crt_probs = evaluate_model(
    fusion_model, test_loader, device, 'Fusion_cRT')
plot_confusion_matrix(crt_labels, crt_preds, 'Fusion_cRT')
save_metrics(crt_metrics, 'Fusion_cRT')

torch.save(fusion_model.state_dict(), MODELS_DIR / 'fusion_crt.pt')
gc.collect()

---
## Step 11: Two-Stage Hierarchical Classification
Stage 1: Hunger vs non-hunger (binary). Stage 2: 4-class fine-grained among non-hunger.

In [ ]:
print('=== Step 11: Hierarchical Two-Stage Model ===')

# Build hierarchical model on top of the trained fusion backbone
hier_backbone = build_student().to(device)
hier_model = HierarchicalModel(hier_backbone, n_fine_classes=4, threshold=0.3).to(device)
print(f'Parameters: {count_parameters(hier_model):,}')

hier_trainer = HierarchicalTrainer(hier_model, device)

best_hier_f1 = 0
for epoch in range(MAX_EPOCHS_STUDENT):
    loss = hier_trainer.train_epoch(train_loader, epoch)
    val_f1 = hier_trainer.validate(val_loader)
    
    if val_f1 > best_hier_f1:
        best_hier_f1 = val_f1
        torch.save(hier_model.state_dict(), MODELS_DIR / 'hierarchical.pt')
    
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d} | Loss={loss:.4f} | Val MacF1={val_f1:.3f}')

# Load best and evaluate
hier_model.load_state_dict(torch.load(MODELS_DIR / 'hierarchical.pt', weights_only=True))

# Evaluate hierarchical model
hier_model.eval()
hier_all_preds = []
hier_all_labels = []
with torch.no_grad():
    for mel, domain, labels in test_loader:
        preds = hier_model.predict(mel.to(device), domain.to(device)).cpu()
        hier_all_preds.extend(preds.numpy())
        hier_all_labels.extend(labels.numpy())

from sklearn.metrics import f1_score, classification_report, accuracy_score, matthews_corrcoef, cohen_kappa_score
hier_all_preds = np.array(hier_all_preds)
hier_all_labels = np.array(hier_all_labels)

hier_metrics = {
    'accuracy': float(accuracy_score(hier_all_labels, hier_all_preds)),
    'macro_f1': float(f1_score(hier_all_labels, hier_all_preds, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(hier_all_labels, hier_all_preds, average='weighted', zero_division=0)),
    'mcc': float(matthews_corrcoef(hier_all_labels, hier_all_preds)),
    'kappa': float(cohen_kappa_score(hier_all_labels, hier_all_preds)),
    'auc_roc': None,
}
print(f'\nHierarchical Model Results:')
print(classification_report(hier_all_labels, hier_all_preds, target_names=CLASSES, zero_division=0))
print(f'  MacF1: {hier_metrics["macro_f1"]:.4f}')
plot_confusion_matrix(hier_all_labels, hier_all_preds, 'Hierarchical')
save_metrics(hier_metrics, 'Hierarchical')
gc.collect()

---
## Step 12: Ablation Study (10 Variants)
Systematically test what each component contributes.

In [ ]:
print('=== Step 12: Ablation Study ===')
print('=' * 60)

ablation_results = {}

# Variants already trained:
ablation_results['full_model'] = crt_metrics
ablation_results['cnn_only'] = cnn_metrics

# Additional variants to train
ablation_variants = [
    ('no_attention', True),
    ('no_bilstm', True),
    ('spec_only', True),
    ('feat_only', True),
]

for variant_name, use_ldam in ablation_variants:
    print(f'\n--- Training: {variant_name} ---')
    model = build_model(variant_name).to(device)
    print(f'  Parameters: {count_parameters(model):,}')
    
    trainer = Trainer(model, device, cls_num_list=train_class_counts,
                      use_ldam=use_ldam, max_epochs=MAX_EPOCHS_STUDENT)
    trainer.fit(train_loader, val_loader, verbose=False)
    
    metrics, preds, labels, probs = evaluate_model(
        model, test_loader, device, variant_name)
    plot_confusion_matrix(labels, preds, variant_name)
    save_metrics(metrics, variant_name)
    ablation_results[variant_name] = metrics
    
    del model, trainer
    gc.collect()

# CE loss variant (same arch as full_model but no LDAM)
print('\n--- Training: ce_loss ---')
ce_model = build_student().to(device)
ce_trainer = Trainer(ce_model, device, cls_num_list=train_class_counts,
                     use_ldam=False, max_epochs=MAX_EPOCHS_STUDENT)
ce_trainer.fit(train_loader, val_loader, verbose=False)
ce_metrics, ce_preds, ce_labels, ce_probs = evaluate_model(
    ce_model, test_loader, device, 'ce_loss')
plot_confusion_matrix(ce_labels, ce_preds, 'ce_loss')
save_metrics(ce_metrics, 'ce_loss')
ablation_results['ce_loss'] = ce_metrics
del ce_model, ce_trainer
gc.collect()

# No augmentation variant — use saved original-only features
print('\n--- Training: no_augmentation ---')
no_aug_model = build_student().to(device)
orig_class_counts = [int((orig_train_labels == i).sum()) for i in range(NUM_CLASSES)]
no_aug_trainer = Trainer(no_aug_model, device, cls_num_list=orig_class_counts,
                         use_ldam=True, max_epochs=MAX_EPOCHS_STUDENT)
orig_train_loader, _ = create_dataloaders(
    orig_train_mel, orig_train_dom, orig_train_labels,
    val_mel, val_dom, val_labels,
    batch_size=BATCH_SIZE, balanced=False,
)
no_aug_trainer.fit(orig_train_loader, val_loader, verbose=False)
noaug_metrics, noaug_preds, noaug_labels, noaug_probs = evaluate_model(
    no_aug_model, test_loader, device, 'no_augmentation')
plot_confusion_matrix(noaug_labels, noaug_preds, 'no_augmentation')
save_metrics(noaug_metrics, 'no_augmentation')
ablation_results['no_augmentation'] = noaug_metrics
del no_aug_model, no_aug_trainer
gc.collect()

# Flat 5-class (already done in step 9, use fusion_metrics before cRT)
ablation_results['flat_5class'] = fusion_metrics

# Hierarchical
ablation_results['hierarchical'] = hier_metrics

In [ ]:
# Print ablation table and comparison
make_ablation_table(ablation_results, phase1_baseline=0.2703)
plot_ablation_comparison(ablation_results)

---
## Step 13: Teacher Model + Knowledge Distillation
Train a wider teacher (~200K params), then distill to the student (~40K params).

In [ ]:
print('=== Step 13a: Train Teacher Model (~200K params) ===')
teacher_model = build_teacher().to(device)
print(f'Teacher parameters: {count_parameters(teacher_model):,}')

teacher_trainer = Trainer(teacher_model, device, cls_num_list=train_class_counts,
                          use_ldam=True, max_epochs=MAX_EPOCHS_TEACHER)
teacher_history = teacher_trainer.fit(train_loader, val_loader)

teacher_metrics, teacher_preds, teacher_labels, teacher_probs = evaluate_model(
    teacher_model, test_loader, device, 'Teacher_200K')
plot_confusion_matrix(teacher_labels, teacher_preds, 'Teacher_200K')
plot_training_history(teacher_history, 'Teacher_200K')
save_metrics(teacher_metrics, 'Teacher_200K')
ablation_results['teacher_200k'] = teacher_metrics

torch.save(teacher_model.state_dict(), MODELS_DIR / 'teacher.pt')
gc.collect()

In [ ]:
print('=== Step 13b: Knowledge Distillation ===')
distilled_student = build_student().to(device)
print(f'Student parameters: {count_parameters(distilled_student):,}')

distilled_student = train_with_distillation(
    teacher_model, distilled_student,
    train_loader, val_loader,
    device, epochs=MAX_EPOCHS_STUDENT,
)

kd_metrics, kd_preds, kd_labels, kd_probs = evaluate_model(
    distilled_student, test_loader, device, 'Distilled_Student')
plot_confusion_matrix(kd_labels, kd_preds, 'Distilled_Student')
save_metrics(kd_metrics, 'Distilled_Student')

torch.save(distilled_student.state_dict(), MODELS_DIR / 'distilled_student.pt')
gc.collect()

---
## Step 14: Quantization & Edge Deployment Analysis

In [ ]:
print('=== Step 14: Quantization ===')

# Quantize the distilled student
quantized = quantize_model(distilled_student)

# Benchmark inference on CPU
sample_mel = torch.randn(1, 1, 64, 438)
sample_dom = torch.randn(1, 32)
print('\nOriginal model (CPU):')
distilled_student.cpu()
benchmark_inference(distilled_student, sample_mel, sample_dom, torch.device('cpu'))

print('\nModel size comparison:')
orig_params = count_parameters(distilled_student)
orig_size_kb = orig_params * 4 / 1024  # FP32
quant_size_kb = orig_params * 1 / 1024  # INT8
print(f'  FP32: {orig_params:,} params = {orig_size_kb:.1f} KB')
print(f'  INT8: {orig_params:,} params ~ {quant_size_kb:.1f} KB')
print(f'  Compression: {orig_size_kb/quant_size_kb:.1f}x')

---
## Step 15: Interpretability
Grad-CAM, temporal attention visualization, and domain feature importance.

In [ ]:
print('=== Step 15a: Grad-CAM Visualizations ===')
# Use the best fusion model
fusion_model.to(device)
try:
    generate_gradcam_gallery(fusion_model, test_loader, device, n_per_class=5)
except Exception as e:
    print(f'  Grad-CAM error (non-critical): {e}')
    print('  Skipping — this can happen with dynamic architectures')

In [ ]:
print('=== Step 15b: Attention Weight Visualization ===')
try:
    plot_attention_weights(fusion_model, test_loader, device, n_per_class=5)
except Exception as e:
    print(f'  Attention viz error (non-critical): {e}')

In [ ]:
print('=== Step 15c: Domain Feature Importance ===')
feature_names = [
    'F0_mean', 'F0_std', 'F0_range', 'Hyperphonation',
    'HNR', 'Jitter', 'Shimmer',
    'MFCC_1', 'MFCC_2', 'MFCC_3', 'MFCC_4', 'MFCC_5', 'MFCC_6',
    'MFCC_7', 'MFCC_8', 'MFCC_9', 'MFCC_10', 'MFCC_11', 'MFCC_12', 'MFCC_13',
    'dMFCC_1', 'dMFCC_2', 'dMFCC_3', 'dMFCC_4', 'dMFCC_5', 'dMFCC_6',
    'dMFCC_7', 'dMFCC_8', 'dMFCC_9', 'dMFCC_10', 'dMFCC_11', 'dMFCC_12',
]

importances = domain_feature_importance(
    fusion_model, test_loader, device, feature_names=feature_names)

print('\nTop 15 most important domain features:')
for i, (fname, imp) in enumerate(importances[:15], 1):
    print(f'  {i:2d}. {fname:<25s}  {imp:+.4f}')

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(20, len(importances))
names = [x[0] for x in importances[:top_n]]
vals = [x[1] for x in importances[:top_n]]
colors = ['#e74c3c' if v > 0 else '#3498db' for v in vals]
ax.barh(range(top_n), vals, color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel('MacF1 Drop (Permutation Importance)')
ax.set_title('Domain Feature Importance')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'domain_feature_importance.png', dpi=150)
plt.show()

---
## Step 16: Hybrid ML+DL Ensemble
Stacking ensemble combining Phase 1 SVM + RF with Phase 2 DL model.

In [ ]:
print('=== Step 16: Hybrid ML+DL Ensemble ===')
print('=' * 60)

# Load Phase 1 features for ML models
from src.feature_extractor import FeatureExtractor
from tqdm import tqdm

print('Extracting 411-dim features for ML models...')
extractor = FeatureExtractor()

def extract_411_features(data_list, desc='Extracting'):
    features = []
    for d in tqdm(data_list, desc=desc):
        try:
            feats, _ = extractor.extract_all(d['audio'])
            features.append(feats)
        except:
            features.append(np.zeros(411, dtype=np.float32))
    return np.array(features, dtype=np.float32)

# We need val and test data with audio — reload if needed
print('Reloading audio for ML feature extraction...')
all_data = load_all_originals()
_, val_data_reload, test_data_reload = group_split(all_data)

val_411 = extract_411_features(val_data_reload, desc='Val 411-dim')
test_411 = extract_411_features(test_data_reload, desc='Test 411-dim')
val_labels_ml = np.array([d['label_idx'] for d in val_data_reload])
test_labels_ml = np.array([d['label_idx'] for d in test_data_reload])

# Load Phase 1 ML models
hybrid = HybridEnsemble()
hybrid.load_ml_models()

if hybrid.ml_models:
    # Get ML probabilities on val and test
    val_ml_probas = hybrid.get_ml_probas(val_411)
    test_ml_probas = hybrid.get_ml_probas(test_411)
    
    # Get DL probabilities
    fusion_model.to(device)
    val_dl_probas = hybrid.get_dl_probas(fusion_model, val_loader, device)
    test_dl_probas = hybrid.get_dl_probas(fusion_model, test_loader, device)
    
    # Fit and evaluate stacking ensemble
    print('\n--- Stacking Ensemble ---')
    hybrid.fit(val_ml_probas, val_dl_probas, val_labels_ml)
    hybrid_preds = hybrid.predict(test_ml_probas, test_dl_probas)
    
    hybrid_metrics = {
        'accuracy': float(accuracy_score(test_labels_ml, hybrid_preds)),
        'macro_f1': float(f1_score(test_labels_ml, hybrid_preds, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(test_labels_ml, hybrid_preds, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(test_labels_ml, hybrid_preds)),
        'kappa': float(cohen_kappa_score(test_labels_ml, hybrid_preds)),
        'auc_roc': None,
    }
    print(f'\nHybrid Stacking Results:')
    print(classification_report(test_labels_ml, hybrid_preds, target_names=CLASSES, zero_division=0))
    print(f'  MacF1: {hybrid_metrics["macro_f1"]:.4f}')
    plot_confusion_matrix(test_labels_ml, hybrid_preds, 'Hybrid_Stacking')
    save_metrics(hybrid_metrics, 'Hybrid_Stacking')
    hybrid.save()
    
    # Weighted average ensemble
    print('\n--- Weighted Average Ensemble ---')
    weighted = WeightedEnsemble()
    weighted.fit(val_ml_probas, val_dl_probas, val_labels_ml)
    weighted_preds = weighted.predict(test_ml_probas, test_dl_probas)
    
    weighted_metrics = {
        'accuracy': float(accuracy_score(test_labels_ml, weighted_preds)),
        'macro_f1': float(f1_score(test_labels_ml, weighted_preds, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(test_labels_ml, weighted_preds, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(test_labels_ml, weighted_preds)),
        'kappa': float(cohen_kappa_score(test_labels_ml, weighted_preds)),
        'auc_roc': None,
    }
    print(f'  MacF1: {weighted_metrics["macro_f1"]:.4f}')
    plot_confusion_matrix(test_labels_ml, weighted_preds, 'Hybrid_Weighted')
    save_metrics(weighted_metrics, 'Hybrid_Weighted')
else:
    print('[WARN] No Phase 1 ML models found. Skipping hybrid ensemble.')
    hybrid_metrics = crt_metrics  # fallback
    weighted_metrics = crt_metrics

del all_data, val_data_reload, test_data_reload
gc.collect()

---
## Step 17: Final Results & Mandatory Ablation Table

In [ ]:
print('\n' + '=' * 70)
print('  PHASE 2 COMPLETE RESULTS')
print('=' * 70)

# All DL models comparison
all_dl_results = {
    'CNN Baseline': cnn_metrics,
    'CNN+BiLSTM+Attn': cba_metrics,
    'LDAM+DRW': ldam_metrics,
    'Fusion+LDAM': fusion_metrics,
    'Fusion+cRT': crt_metrics,
    'Hierarchical': hier_metrics,
    'Teacher (200K)': teacher_metrics,
    'Distilled': kd_metrics,
}

print(f"\n  {'Model':<22s} {'Acc':>7s} {'MacF1':>7s} {'WgtF1':>7s} {'MCC':>7s}")
print('  ' + '-' * 48)

# Phase 1 baseline
print(f"  {'Phase1 SVM (baseline)':<22s} {'0.8152':>7s} {'0.2703':>7s} {'0.7831':>7s} {'0.2156':>7s}")
print('  ' + '-' * 48)

best_name = ''
best_f1 = 0
for name, m in all_dl_results.items():
    print(f"  {name:<22s} {m['accuracy']:>7.4f} {m['macro_f1']:>7.4f} "
          f"{m['weighted_f1']:>7.4f} {m['mcc']:>7.4f}")
    if m['macro_f1'] > best_f1:
        best_f1 = m['macro_f1']
        best_name = name

print(f'\n  Best DL model: {best_name} (MacF1 = {best_f1:.4f})')
print(f'  Improvement over Phase 1: {best_f1 - 0.2703:+.4f}')

In [ ]:
# Mandatory 3-model comparison
ml_baseline = {
    'accuracy': 0.8152, 'macro_f1': 0.2703,
    'weighted_f1': 0.7831, 'mcc': 0.2156,
}

# Best DL-only model (before hybrid)
dl_only = crt_metrics

# Best hybrid
best_hybrid = max([hybrid_metrics, weighted_metrics],
                  key=lambda m: m['macro_f1'])

three_model_comparison(ml_baseline, dl_only, best_hybrid)

In [ ]:
# Full ablation table
make_ablation_table(ablation_results, phase1_baseline=0.2703)
plot_ablation_comparison(ablation_results)

In [ ]:
# Save all results to Drive
final_results = {
    'phase1_baseline': ml_baseline,
    'dl_models': {k: v for k, v in all_dl_results.items()},
    'hybrid_stacking': hybrid_metrics,
    'hybrid_weighted': weighted_metrics,
    'ablation': ablation_results,
}

with open(RESULTS_DIR / 'phase2_final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=str)
print(f'\nAll results saved to {RESULTS_DIR}/')

---
## Step 18: Verify Drive Persistence

In [ ]:
print('Files saved on Google Drive:')
print('=' * 60)
for folder in ['src/phase2', 'features', 'models/phase2',
               'results/phase2/metrics', 'results/phase2/plots']:
    p = Path(PROJECT_DIR) / folder
    if p.exists():
        count = len(list(p.iterdir()))
        print(f'  {folder + "/":40s} {count} files')

print(f'\nPhase 2 pipeline complete.')
print(f'Best DL model: {best_name} (MacF1 = {best_f1:.4f})')
print(f'Phase 1 baseline: SVM (MacF1 = 0.2703)')
print(f'Improvement: {best_f1 - 0.2703:+.4f}')